In [ ]:
import os

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import gymnasium as gym
import numpy as np
from pop import AI, Util

# CartPole DQN은 작은 네트워크이므로 CPU 저메모리 정책을 사용한다.
AI.configure("cpu")
dqn = AI.DQN(state_size=4, hidden_size=8, output_size=1)

env = gym.make('CartPole-v1', render_mode='rgb_array')
print("Gymnasium:", gym.__version__)
print("device:", AI.device_policy())


In [ ]:
for episode in range(1000):
    state, info = env.reset(seed=42 + episode)
    total_reward = 0.0
    states, rewards, actions = [], [], []

    while True:
        frame = env.render()
        Util.imshow('CartPole', frame, width=600, height=400, mode='RGB')
        # np.bool 기반 구형 DQN.run 대신 확률을 받아 명시적으로 행동을 선택한다.
        prediction = dqn.model.predict(
            np.asarray([state], dtype=np.float32),
            verbose=0,
        )
        action = int(prediction[0, 0] >= 0.5)

        states.append(np.asarray(state, dtype=np.float32))
        actions.append([float(action)])

        next_state, reward, terminated, truncated, info = env.step(action)
        rewards.append(float(reward))
        total_reward += reward
        state = next_state

        if terminated or truncated:
            loss = dqn.train(states, rewards, actions)
            print(
                "episode", episode + 1,
                "steps", len(rewards),
                "reward", total_reward,
                "loss", loss,
            )
            break

env.close()
